# Medicine Variant Identity Verification

This notebook verifies whether medicine identity in the repository depends on `unit` alone or also includes numeric size/weight fields. It will inspect schema definitions, query the database, and review relevant code paths for duplicate detection, purchase stock, sale stock deduction, product search labels, company profile stock, and historical `TransactionItem.productId` usage.

**Goal:** Confirm that variants like `Napa 100 g Company A` and `Napa 250 g Company A` are treated as separate products and not merged by `unit` alone.

## Section 1: Inspect Prisma Schema for Medicine/Product Fields

Load the Prisma schema and identify product-related fields. Check whether there is an existing numeric weight/size field in addition to unit.

from pathlib import Path
schema_path = Path('prisma/schema.prisma')
print(f'Prisma schema path: {schema_path.resolve()}')
schema_text = schema_path.read_text()
start = schema_text.find('model Product')
end = schema_text.find('@@index([companyId])', start)
print(schema_text[start:end+len('@@index([companyId])')])

## Section 2: Query Existing Medicine Product Records

Query the database for medicine products and inspect fields such as name, unit, weight/size, company, and any related product variant data.

import json
from pathlib import Path
from prisma import Prisma
prisma = Prisma()

async def inspect_products():
    await prisma.connect()
    try:
        meds = await prisma.product.find_many(
            where={"productType": "MEDICINE"},
            select={
                "id": True,
                "code": True,
                "name": True,
                "unit": True,
                "companyId": True,
                "defaultPurchasePrice": True,
                "defaultSellingPrice": True,
                "stockBalance": {"select": {"quantityOnHand": True}},
            },
            take=200,
        )
        grouping = await prisma.product.group_by(
            by=["name", "unit", "companyId"],
            _count={"_all": True},
            order={"_count": {"_all": "desc"}},
            take=100,
        )
        unit_grouping = await prisma.product.group_by(
            by=["unit"],
            _count={"_all": True},
            order={"_count": {"_all": "desc"}},
        )
    finally:
        await prisma.disconnect()
    return {
        "meds": meds,
        "grouping": grouping,
        "unit_grouping": unit_grouping,
    }

result = __import__('asyncio').run(inspect_products())
print(json.dumps(result, default=str, indent=2)[:20000])
Path('tmp/medicine_variant_identity_data.json').write_text(json.dumps(result, default=str, indent=2))
print('Saved query results to tmp/medicine_variant_identity_data.json')

## Section 3: Determine `unit` Field Semantics

Analyze product records to confirm whether `unit` stores numeric weight/size or only measurement labels like `g`, `kg`, `piece`.

## Section 4: Verify Medicine Duplicate Detection Logic

Review and test creation logic to ensure duplicate detection uses product name, numeric size, unit, and company appropriately.

from pathlib import Path
import re
paths = [
    Path('src/features/products/actions.ts'),
    Path('src/features/purchases/actions.ts'),
]
for path in paths:
    print(f'--- {path} ---')
    text = path.read_text()
    for term in ['findFirst', 'where:', 'unit', 'name:', 'company', 'productType', 'productId', 'duplicate']:
        if term in text:
            print(f'Found {term} in {path}')
    snippet = re.search(r'(existingProduct.*?findFirst.*?)', text, re.S)
    if snippet:
        print('existingProduct snippet:')
        print(snippet.group(1)[:1200])
    print()

## Section 5: Verify Purchase Stock Tracking

Test medicine purchase flows to ensure stock is recorded per distinct variant and not merged based on unit alone.

from pathlib import Path
text = Path('src/features/purchases/actions.ts').read_text()
start = text.find('const existingProduct')
end = text.find('resolvedItems.push', start)
print(text[start:end+2000])
print('

Stock update logic snippet:')
start2 = text.find('for (const [productId, quantity] of productQuantities.entries())', text)
end2 = text.find('return {', start2)
print(text[start2:end2])

## Section 6: Verify Sale Stock Deduction

Test medicine sale flows to confirm stock deduction applies to the correct variant based on name, size, unit, and company.

from pathlib import Path
text = Path('src/features/sales/actions.ts').read_text()
start = text.find('const products = await tx.product.findMany')
end = text.find('for (const [productId, quantity] of productQuantities.entries())', start)
print(text[start:end+500])
print('

Sale stock movement snippet:')
start2 = text.find('for (const [productId, quantity] of productQuantities.entries())', text)
end2 = text.find('return {', start2)
print(text[start2:end2])

## Section 7: Verify Product Search and Dropdown Labels

Inspect product search and dropdown label generation to ensure variants are displayed distinctly with size and company.

from pathlib import Path
from glob import glob
files = glob('src/**/*search*.{ts,tsx}', recursive=True) + glob('src/**/*dropdown*.{ts,tsx}', recursive=True)
for file in sorted(set(files)):
    print(file)
    text = Path(file).read_text()
    if 'unit' in text or 'label' in text or 'name' in text:
        print(' snippet:')
        print('\n'.join([line for line in text.splitlines() if 'unit' in line or 'label' in line or 'name' in line][:40]))
    print()

## Section 8: Verify Company Profile Stock and Historical Transaction Links

Check company profile stock summaries and historical `TransactionItem.productId` references to ensure variant identity is preserved without altering past data.

from pathlib import Path
paths = [
    Path('src/app/dashboard/companies/[id]/page.tsx'),
    Path('src/app/dashboard/companies/page.tsx'),
    Path('src/components/dashboard/stock/add-stock-modal.tsx'),
]
for path in paths:
    print(f'--- {path} ---')
    text = path.read_text()
    print('\n'.join([line for line in text.splitlines() if 'product' in line.lower() or 'stock' in line.lower()][:80]))
    print()

## Section 9: Run TypeScript Compilation and Build

Execute `npx tsc --noEmit` and `npm run build` to validate the codebase after the fix and confirm no type or build errors.

import json
import subprocess
from pathlib import Path
commands = [
    ['npx', 'tsc', '--noEmit'],
    ['npm', 'run', 'build'],
]
results = {}
for cmd in commands:
    print('Running:', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    results[' '.join(cmd)] = {
        'returncode': proc.returncode,
        'stdout': proc.stdout[:4000],
        'stderr': proc.stderr[:4000],
    }
    print('Return code:', proc.returncode)
    print(proc.stdout[:2000])
    print(proc.stderr[:2000])
Path('tmp/medicine_variant_build_results.json').write_text(json.dumps(results, indent=2))
print('Saved build results to tmp/medicine_variant_build_results.json')